In [ ]:
print("Hello World!")

In [2]:
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

True

In [ ]:
import nest_asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient

nest_asyncio.apply()

# Connect to the mcp-time server for timezone-aware operations
# This Go-based server provides tools for current time, relative time parsing,
# timezone conversion, duration arithmetic, and time comparison
mcp_client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "@theo.foobar/mcp-time"],
        }
    },
)

# Load tools from the MCP server
mcp_tools = await mcp_client.get_tools()
print(f"Loaded {len(mcp_tools)} MCP tools: {[t.name for t in mcp_tools]}")

Loaded 5 MCP tools: ['add_time', 'compare_time', 'convert_timezone', 'current_time', 'relative_time']


In [9]:
from langchain.agents import create_agent

agent_with_mcp = create_agent(
    model="claude-haiku-4-5",
    tools=mcp_tools,
    system_prompt="You are a helpful assistant",
)

In [12]:
result = await agent_with_mcp.ainvoke(
    {"messages": [{"role": "user", "content": "What's the time in Vilnius right now?"}]}
)
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What's the time in Vilnius right now?
================================== Ai Message ==================================

[{'id': 'toolu_018xQikh8NCwh38bRERDxZa8', 'caller': {'type': 'direct'}, 'input': {'timezone': 'Europe/Vilnius'}, 'name': 'current_time', 'type': 'tool_use'}]
Tool Calls:
  current_time (toolu_018xQikh8NCwh38bRERDxZa8)
 Call ID: toolu_018xQikh8NCwh38bRERDxZa8
  Args:
    timezone: Europe/Vilnius
================================= Tool Message =================================
Name: current_time

[{'type': 'text', 'text': '2026-04-07T00:17:13+03:00', 'id': 'lc_c9da0ba6-9d38-46f3-86d6-f32485af1ce4'}]
================================== Ai Message ==================================

The current time in Vilnius is **12:17 AM** (00:17) on **April 7, 2026** (UTC+3).
